In [51]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [52]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [53]:
# We create a generator for randomness required in this protocol
def get_quantum_random_bits(n):
  """Generates n random bits by measuring qubits in superposition."""
  qc = QuantumCircuit(n)
  for i in range(n):
      qc.h(i)
  qc.measure_all()

  backend = BasicSimulator()
  # Transpiling for the simulator to ensure compatibility
  job = backend.run(transpile(qc, backend), shots=1)
  result = job.result().get_counts()

  # Get the resulting bitstring (e.g., '1011...')
  bitstring = list(result.keys())[0]
  # Return as a list of integers, reversed to match Qiskit qubit ordering
  return [int(b) for b in bitstring[::-1]]

In [54]:
num_bits = 16
# Generating all required random choices using quantum measurement [cite: 3, 8]
alice_bits = get_quantum_random_bits(num_bits)   # Alice's 16 classical bits
alice_bases = get_quantum_random_bits(num_bits)  # 0: Z (Rectilinear), 1: X (Diagonal)
bob_bases = get_quantum_random_bits(num_bits)    # Bob's random choice of bases

In [55]:
# --- PHASE 1: ALICE PREPARES QUBITS ---
# Alice encodes her bits into qubits based on her chosen bases
quantum_channel = []

for i in range(num_bits):
  qc = QuantumCircuit(1, 1)

  # Encode the bit
  if alice_bits[i] == 1:
      qc.x(0)

  # Apply basis transformation
  if alice_bases[i] == 1: # Diagonal basis (X)
      qc.h(0)
  # else: Rectilinear (Z) - no gate needed (|0> or |1>)

  quantum_channel.append(qc)

In [56]:
# --- PHASE 2: BOB MEASURES QUBITS ---
# Bob receives the qubits and measures them in his chosen bases
bob_results = []

for i in range(num_bits):
  qc = quantum_channel[i]

  # Bob applies basis transformation before measuring
  if bob_bases[i] == 1: # Bob chooses Diagonal
      qc.h(0)

  qc.measure(0, 0)

  # Simulate the measurement
  backend = BasicSimulator()
  job = backend.run(transpile(qc, backend), shots=1)
  result = job.result().get_counts()
  measured_bit = int(list(result.keys())[0])
  bob_results.append(measured_bit)

In [57]:
# --- PHASE 3: SIFTING (Public Channel) ---
# Alice and Bob compare bases and keep bits where bases matched
shared_key = []
ismatching = []

for i in range(num_bits):
  if alice_bases[i] == bob_bases[i]:
      ismatching.append("Y")
      shared_key.append(alice_bits[i])
  else:
      ismatching.append("")


In [58]:
# Helper function to format qubit state
def get_qubit_state(bits, bases):
  qubit_state = []
  if len(bits) != len(bases):
    return "Invalid input"
  for i in range(len(bits)):
    if bases[i]==0:
      qubit_state.append(f"0") if bits[i]==0 else qubit_state.append(f"1")
    else:
      qubit_state.append(f"+") if bits[i]==0 else qubit_state.append(f"-")

  return qubit_state

qubit_state = get_qubit_state(alice_bits, alice_bases)

In [59]:
# Helper function to format the table
def format_as_table(headers, data_lists):
  """
  Formats lists into a horizontal table where headers are the first
  element of each row.
  """
  if not headers or not data_lists:
    return ""

  # 1. Determine the width needed for the header column
  header_width = max(len(h) for h in headers)

  # 2. Determine the width for each data cell to ensure alignment
  # We flatten the lists to find the longest string representation of any data point
  all_items = [str(item) for sublist in data_lists for item in sublist]
  cell_width = max(len(item) for item in all_items) if all_items else 1

  table_lines = []

  # 3. Build each row
  for header, row_data in zip(headers, data_lists):
    # Start the row with the header label
      row_str = f"{header:<{header_width}} |"
      # Add each data point from the list to the row
      for item in row_data:
        if header == "Alice's bases" or header == "Bob's bases":
          item = "s" if item == 0 else "d"
        row_str += f" {str(item):<{cell_width}} |"
      table_lines.append(row_str)
      if header == "Index":
        table_lines.append("-" * len(row_str))

  return "\n".join(table_lines)

In [60]:
# --- RESULTS ---
indices = list(range(len(alice_bits)))
results = [indices, alice_bits, alice_bases, qubit_state, bob_bases, bob_results, ismatching]
headers=["Index",
  "Alice's initial bits",
  "Alice's bases",
  "Qubit state",
  "Bob's bases",
  "Bob's results",
  "Matching bases (Y if match)"]
print(format_as_table(headers, results))
print("-" * 40)
print(f"Final Shared Key:  {shared_key}")

Index                       | 0  | 1  | 2  | 3  | 4  | 5  | 6  | 7  | 8  | 9  | 10 | 11 | 12 | 13 | 14 | 15 |
-------------------------------------------------------------------------------------------------------------
Alice's initial bits        | 0  | 1  | 0  | 0  | 1  | 0  | 0  | 0  | 1  | 1  | 1  | 1  | 1  | 0  | 1  | 1  |
Alice's bases               | s  | s  | s  | d  | s  | s  | s  | d  | d  | d  | d  | d  | s  | s  | s  | s  |
Qubit state                 | 0  | 1  | 0  | +  | 1  | 0  | 0  | +  | -  | -  | -  | -  | 1  | 0  | 1  | 1  |
Bob's bases                 | d  | d  | s  | d  | s  | d  | d  | d  | s  | d  | s  | d  | d  | s  | d  | s  |
Bob's results               | 0  | 0  | 0  | 0  | 1  | 1  | 1  | 0  | 0  | 1  | 1  | 1  | 0  | 0  | 0  | 1  |
Matching bases (Y if match) |    |    | Y  | Y  | Y  |    |    | Y  |    | Y  |    | Y  |    | Y  |    | Y  |
----------------------------------------
Final Shared Key:  [0, 0, 1, 0, 1, 1, 0, 1]
